## Детекция клеток

Ваша задача: обучить YOLO для детекции дрожжевых клеток и микроструктур (см. [07_object_detection.ipynb](../workshops/07_object_detection.ipynb)). Всё необходимое для запуска обучения вы можете взять из ноутбука с практикой, доделать нужно будет самую малость:
- реализовать расчёт Mean Average Precision для всего валидационного сета
- попробовать привести regression loss к виду, который используется в Yolo9000 и YoloV3
- подобрать лучшие размеры якорных рамок с помощью кластеризации

Основная цель: любыми средствами добиться $mAP > 0.6$ на валидации.

Используйте класс `torchmetrics.detection.MeanAveragePrecision` для расчёта $mAP$.

Нас будет интересовать именно значение `map` в словаре со всеми метриками - это mean average precision, усреднённый по всем отсечкам intersection over union в диапазоне $[0.5, 0.95]$ (см. документацию к классу).

При решении можно пользоваться `lightning` или писать цикл обучения вручную. В последнем случае не забудьте вручную отправить модель и батчи на GPU, чтобы обучалось быстрее

In [86]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.figure import Figure
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.functional.detection import intersection_over_union
from torchvision.ops.boxes import box_convert

In [3]:
def process_yolo_preds(preds: Tensor, rescaled_anchors: Tensor) -> tuple[Tensor, Tensor, Tensor]:
    """
    Преобразование выходов модели в
    1. Логит наличия объекта (вероятность получается применением сигмоиды)
    2. Положение рамки относительно ячейки в формате cxcywh
    3. Логиты классов (вероятности получаются применением softmax)
    """
    rescaled_anchors = rescaled_anchors.view(1, len(rescaled_anchors), 1, 1, 2)
    box_predictions = preds[..., 1:5].clone()

    box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
    box_predictions[..., 2:] = torch.exp(box_predictions[..., 2:]) * rescaled_anchors

    scores = preds[..., 0:1]
    return scores, box_predictions, preds[..., 5:]

In [61]:
GRID_SIZE = 8
IMAGE_SIZE = 256

# Для 1-го и 2-го заданий я использовал именно эти якоря, точнее, первый из них - [48, 72].
# Все результаты и ячейки до 4 задания оставлены при использовании этого якоря.
# ANCHORS = [ 
#     [48, 72],
#     # [64, 64],
#     # [72, 48],
# ]

# Эти 5 якорей нашлись уже в 3 задании по результатам кластеризации.
# Перенес сюда, чтобы не копировать много кода вниз
ANCHORS = [
    [43, 48],
    [57, 85],
    [62, 62],
    [70, 106],
    [91, 88]
]

rescaled_anchors = torch.tensor(ANCHORS) / IMAGE_SIZE * GRID_SIZE

In [87]:
class CNNBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(0.1)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return self.activation(x)

# Тут поменял num_anchors на 5 и сделал глубже
class TinyYOLO(nn.Module):
    def __init__(self, num_classes: int = 2, num_anchors: int = 5, in_channels: int = 1) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.num_anchors = num_anchors
        self.layers = nn.Sequential(
            CNNBlock(1, 16, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(16, 32, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(32, 64, kernel_size=3, stride=2, padding=1, dilation=2),
            CNNBlock(64, 128, kernel_size=3, stride=2, padding=1, groups=8),
            CNNBlock(128, 256, kernel_size=3, stride=1, padding=1, groups=8),
            CNNBlock(256, 512, kernel_size=3, stride=1, padding=1, groups=16),
            CNNBlock(512, 512, kernel_size=3, stride=1, padding=1, groups=16),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(512, num_anchors * (num_classes + 5), kernel_size=1)
        )
    
    def forward(self, x: Tensor) -> Tensor:
        x = self.layers(x)
        B, _, W, H = x.shape
        x = x.view(B, self.num_anchors, self.num_classes + 5, W, H)  # B A F W H
        x = x.permute(0, 1, 3, 4, 2)  # B A W H F
        return x


model = TinyYOLO(in_channels=1, num_anchors=1, num_classes=2)
print(sum(p.numel() for p in model.parameters()))
model.forward(torch.randn(2, 1, 256, 256)).shape

297079


torch.Size([2, 1, 8, 8, 7])

In [88]:
def iou_wh(wh1: Tensor, wh2: Tensor) -> Tensor:
    # IoU based on width and height of bboxes

    # intersection
    intersection_area = torch.min(wh1[..., 0], wh2[..., 0]) * torch.min(wh1[..., 1], wh2[..., 1])

    # union
    box1_area = wh1[..., 0] * wh1[..., 1]
    box2_area = wh2[..., 0] * wh2[..., 1]
    union_area = box1_area + box2_area - intersection_area

    iou_score = intersection_area / union_area
    return iou_score


def boxes_to_cells(
    boxes: Tensor,
    classes: Tensor,
    rescaled_anchors: Tensor,
    grid_size: int = 8,
    ignore_iou_thresh: float = 0.5,
) -> Tensor:
    """
    Переводит bbox представление в клеточное представление, где каждая рамка -
    (id класса, вероятность нахождения объекта, cx, cy, w, h), а клеточное представление
    имеет размер (batch_size, n_anchors, grid_size, grid_size, 6), в последней размерности
    хранятся признаки ячейки: класс объекта, вероятность объекта, координаты и размеры рамки
    относительно ячейки

    Args:
        boxes (Tensor): тензор со всеми рамками
        classes (Tensor): тензор с id классов объектов
        rescaled_anchors (Tensor): тензор размера (n_anchors, 2) с размерами якорей в долях от размеров ячейки
        grid_size (int): размер сетки,
        ignore_iou_thresh (float, optional): значение IoU для рамок, при котором ячейка,
            занятая более чем одним объектом, будет специально помечена для игнорирования
    """
    targets = torch.zeros((len(rescaled_anchors), grid_size, grid_size, 6))

    # Каждой рамке сопоставляем клетку и наиболее подходящий якорь
    for box, class_label in zip(boxes, classes):
        iou_anchors = iou_wh(box[2:4], rescaled_anchors / grid_size)
        anchor_indices = iou_anchors.argsort(descending=True, dim=0)
        x, y, width, height = box

        # Относим рамку к наиболее подходящему якорю
        has_anchor = False
        for anchor_idx in anchor_indices:
            s = grid_size

            # Определяем клетку, к которой относится рамка
            i, j = int(s * y), int(s * x)
            anchor_taken = targets[anchor_idx, i, j, 0]

            # Проверяем, доступна ли якорная рамка для текущей ячейки
            if not anchor_taken and not has_anchor:
                # Пересчитываем координаты по отношению к клетке
                x_cell, y_cell = s * x - j, s * y - i
                width_cell, height_cell = (width * s, height * s)
                box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])

                # Заполняем содержимое для выбранной клетки
                targets[anchor_idx, i, j, 0] = 1  # указатель, что в клетке есть объект
                targets[anchor_idx, i, j, 1:5] = box_coordinates
                targets[anchor_idx, i, j, 5] = int(class_label)

                has_anchor = True

            # Если якорь уже выбран, проверим IoU, если больше threshold - пометим клетку -1
            elif not anchor_taken and iou_anchors[anchor_idx] > ignore_iou_thresh:
                targets[anchor_idx, i, j, 0] = -1

    return targets

In [89]:
class YeastDetectionDataset(Dataset):
    def __init__(
        self, subset_dir: Path, anchors: list[tuple[int, int]], image_size: int, grid_size: int = 8
    ) -> None:
        super().__init__()
        self.subset_dir = subset_dir
        self.items = list((self.subset_dir / "inputs").glob("*.pt"))
        # Ignore IoU threshold
        self.ignore_iou_thresh = 0.5
        self.rescaled_anchors = torch.tensor(anchors) / image_size * grid_size
        self.grid_size = grid_size
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        image_path = self.items[index]
        # load everything
        image = torch.load(image_path, weights_only=True).unsqueeze(0)
        classes = (
            torch.load(self.subset_dir / "classes" / image_path.parts[-1], weights_only=True) + 1
        )
        boxes = torch.load(
            self.subset_dir / "bounding_boxes" / image_path.parts[-1], weights_only=True
        )
        boxes = box_convert(boxes, "xyxy", "cxcywh") / self.image_size

        # convert boxes to cells
        targets = boxes_to_cells(
            boxes, classes, self.rescaled_anchors, self.grid_size, self.ignore_iou_thresh
        )
        return image, targets

In [90]:
CLASS_LABELS = ["trap", "cell"]
train_dataset = YeastDetectionDataset(
    Path("yeast_cell_in_microstructures_dataset/train"), anchors=ANCHORS, image_size=256
)
val_dataset = YeastDetectionDataset(Path("yeast_cell_in_microstructures_dataset/val"), anchors=ANCHORS, image_size=256)

batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

In [9]:
x, y = train_dataset[2]
y.shape

torch.Size([1, 8, 8, 6])

In [91]:
def cells_to_bboxes(cells: Tensor, rescaled_anchors: Tensor, is_predictions=False) -> Tensor:
    """
    Переводит клеточное представление в bbox представление, где каждая рамка -
    (id класса, вероятность нахождения объекта, cx, cy, w, h), а клеточное представление
    имеет размер (batch_size, n_anchors, grid_size, grid_size, 6), в последней размерности
    хранятся признаки ячейки: класс объекта, вероятность объекта, координаты и размеры рамки
    относительно ячейки

    Args:
        cells (Tensor): тензор размера (batch_size, n_anchors, width, height, 6)
        rescaled_anchors (Tensor): тензор размера (n_anchors, 2) с размерами якорей в долях от размеров ячейки
        is_predictions (bool, optional): являются ли входные ячейки предсказаниями или верной аннотацией.
    """

    if is_predictions:
        scores, box_predictions, logits = process_yolo_preds(cells, rescaled_anchors)
        scores = torch.sigmoid(scores)
        best_class = torch.argmax(logits, dim=-1).unsqueeze(-1) + 1

    else:
        box_predictions = cells[..., 1:5].clone()
        scores = cells[..., 0:1]
        best_class = cells[..., 5:6]

    # масштабируем размер рамок [0, grid_size] -> [0, 1]
    _, _, H, W, _ = cells.shape
    range_y, range_x = torch.meshgrid(
        torch.arange(H, dtype=cells.dtype, device=cells.device),
        torch.arange(W, dtype=cells.dtype, device=cells.device),
        indexing="ij",
    )
    x = torch.cat(
        [
            best_class,
            scores,
            (box_predictions[..., 0:1] + range_x[None, None, :, :, None]) / W,  # X center
            (box_predictions[..., 1:2] + range_y[None, None, :, :, None]) / H,  # Y center
            box_predictions[..., 2:3] / W,  # Width
            box_predictions[..., 3:4] / H,  # Height
        ],
        -1,
    )

    return x.view(-1, 6)

In [92]:
def plot_image(image: Tensor, boxes: Tensor, class_labels: list[str]) -> Figure:
    # назначим цвета для классов
    colour_map = plt.get_cmap("tab20b")
    colors = [colour_map(i) for i in np.linspace(0, 1, len(class_labels))]

    fig, ax = plt.subplots(1)
    ax.imshow(image, cmap="gray")
    h, w = image.shape
    for box in boxes:
        # добавляем прямоугольник
        class_pred = box[0] - 1
        box = box[2:]
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2

        rect = patches.Rectangle(
            (upper_left_x * w, upper_left_y * h),
            box[2] * w,
            box[3] * h,
            linewidth=2,
            edgecolor=colors[int(class_pred)],
            facecolor="none",
        )
        ax.add_patch(rect)

        # добавляем подпись
        ax.text(
            upper_left_x * w,
            upper_left_y * h,
            s=class_labels[int(class_pred)],
            color="white",
            verticalalignment="top",
            bbox={"color": colors[int(class_pred)], "pad": 0},
        )

    return fig

In [ ]:
rescaled_anchors = torch.tensor(ANCHORS) / IMAGE_SIZE * GRID_SIZE

# Преобразуем клетки к рамкам
boxes = cells_to_bboxes(y.unsqueeze(0), rescaled_anchors)

# Оставим только рамки с объектами и нарисуем
boxes = boxes[boxes[:, 1] == 1]
fig = plot_image(x[0].to("cpu"), boxes, CLASS_LABELS)
plt.show()

In [93]:
class YOLOLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, pred: Tensor, target: Tensor, anchors: Tensor) -> Tensor:
        # ниже входные тензоры будут меняться на месте, так что склонируем их
        pred = pred.clone()
        target = target.clone()

        # разделяем рамки на содержащие объекты и не содержащие
        # NB: ещё могут быть -1, куда отнеслось более 1 объекта - их не учитываем
        obj = target[..., 0] == 1
        no_obj = target[..., 0] == 0

        # преобразуем предсказания bbox
        scores, pred_boxes, logits = process_yolo_preds(pred, anchors)

        # no object loss: кросс-энтропия вместо MSE
        no_object_loss = self.bce(
            (scores[no_obj]),
            (target[..., 0:1][no_obj]),
        )

        # object loss: учим предсказывать IoU
        ious = intersection_over_union(pred_boxes[obj], target[..., 1:5][obj]).detach()
        object_loss = self.mse(scores[obj].sigmoid(), ious * target[..., 0:1][obj])

        # box coordinate loss: логарифмируем размеры рамок перед расчётом MSE
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        pred[..., 1:3] = pred[..., 1:3].sigmoid()
        target[..., 3:5] = torch.log(1e-6 + target[..., 3:5] / anchors)
        box_loss = self.mse(pred[..., 1:5][obj], target[..., 1:5][obj])

        # class loss: здесь всё обычно
        class_loss = self.cross_entropy(logits[obj], target[..., 5][obj].long() - 1)

        # Total loss
        return box_loss + object_loss + no_object_loss + class_loss

In [14]:
torch.manual_seed(42)
x, y = next(iter(train_loader))
model = TinyYOLO(in_channels=1, num_classes=2, num_anchors=len(rescaled_anchors))
print(sum(p.numel() for p in model.parameters()))

preds = model.forward(x)

109431


In [15]:
loss_fn = YOLOLoss()
loss_fn.forward(preds, y, rescaled_anchors)

tensor(1.6522, grad_fn=<AddBackward0>)

In [16]:
optim = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)

In [17]:
for _ in range(20):
    preds = model.forward(x)
    loss = loss_fn(preds, y, rescaled_anchors)
    print(loss)
    loss.backward()
    optim.step()
    optim.zero_grad()

tensor(1.6522, grad_fn=<AddBackward0>)
tensor(1.0784, grad_fn=<AddBackward0>)
tensor(0.7205, grad_fn=<AddBackward0>)
tensor(0.5791, grad_fn=<AddBackward0>)
tensor(0.4934, grad_fn=<AddBackward0>)
tensor(0.4047, grad_fn=<AddBackward0>)
tensor(0.3356, grad_fn=<AddBackward0>)
tensor(0.3264, grad_fn=<AddBackward0>)
tensor(0.2968, grad_fn=<AddBackward0>)
tensor(0.2564, grad_fn=<AddBackward0>)
tensor(0.2487, grad_fn=<AddBackward0>)
tensor(0.2237, grad_fn=<AddBackward0>)
tensor(0.1983, grad_fn=<AddBackward0>)
tensor(0.1890, grad_fn=<AddBackward0>)
tensor(0.1838, grad_fn=<AddBackward0>)
tensor(0.1630, grad_fn=<AddBackward0>)
tensor(0.1524, grad_fn=<AddBackward0>)
tensor(0.1505, grad_fn=<AddBackward0>)
tensor(0.1321, grad_fn=<AddBackward0>)
tensor(0.1200, grad_fn=<AddBackward0>)


In [ ]:
idx = 2
boxes = cells_to_bboxes(preds[idx].detach().unsqueeze(0), rescaled_anchors, is_predictions=True)
boxes = boxes[boxes[:, 1] > 0.2]
fig = plot_image(x[idx, 0].to("cpu"), boxes, CLASS_LABELS)
plt.show()

In [ ]:
from torchvision.ops.boxes import nms

boxes = boxes[nms(box_convert(boxes[:, 2:], "cxcywh", "xyxy"), boxes[:, 1], iou_threshold=0.2)]
fig = plot_image(x[idx, 0].to("cpu"), boxes, CLASS_LABELS)
plt.show()

In [20]:
from pprint import pprint
from torchmetrics.detection import MeanAveragePrecision

metric = MeanAveragePrecision(iou_type="bbox", box_format="cxcywh", class_metrics=True)
predicted_boxes = []
target_boxes = []
# итерируемся по элементам батча, собирая пердсказанные и верные рамки
for i in range(len(y)):
    # получаем предсказанные рамки
    pred_boxes = cells_to_bboxes(preds[i:i+1], rescaled_anchors, is_predictions=True)
    # ВАЖНО: делаем non-max suppression ДО расчёта метрик
    pred_boxes = pred_boxes[nms(box_convert(pred_boxes[:, 2:], "cxcywh", "xyxy"), pred_boxes[:, 1], iou_threshold=0.3)]
    predicted_boxes.append(
        dict(
            boxes=pred_boxes[:, 2:],
            scores=pred_boxes[:, 1],
            labels=pred_boxes[:, 0].long(),
        )
    )
    # достаём правильные рамки
    true_boxes = cells_to_bboxes(y[i:i+1], rescaled_anchors)
    true_boxes = true_boxes[true_boxes[:, 1] == 1]

    target_boxes.append(
        dict(
            boxes=true_boxes[:, 2:],
            labels=true_boxes[:, 0].long(),
        )
    )

# обновляем состояние метрики
# можно было и сразу посчитать, но мы имитируем ситуацию,
# когда метрики используется для сбора данных по всем батчам
metric.update(predicted_boxes, target_boxes)

pprint(metric.compute())

{'classes': tensor([1, 2], dtype=torch.int32),
 'map': tensor(0.5181),
 'map_50': tensor(0.9558),
 'map_75': tensor(0.4785),
 'map_large': tensor(-1.),
 'map_medium': tensor(-1.),
 'map_per_class': tensor([0.5942, 0.4421]),
 'map_small': tensor(0.5181),
 'mar_1': tensor(0.2807),
 'mar_10': tensor(0.5835),
 'mar_100': tensor(0.5835),
 'mar_100_per_class': tensor([0.6500, 0.5169]),
 'mar_large': tensor(-1.),
 'mar_medium': tensor(-1.),
 'mar_small': tensor(0.5835)}


### Задание 1 (3 балла). Цикл обучения с расчётом Mean Average Precision

Запустите обучение модели из практики на всём обучающем датасете, выведите значение $mAP$ на валидационном датасете после окончания обучения.

В этом задании добейтесь $mAP > 0.3$, если всё сделано правильно - для этого должно хватать 30-50 эпох.

In [94]:
def calculate_val_metrics(model, val_loader, anchors, device):
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", box_format="cxcywh", class_metrics=True)
    
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            
            predictions = model(images)
            
            predicted_boxes = []
            target_boxes = []
            
            for i in range(len(targets)):
                pred_boxes_batch = cells_to_bboxes(
                    predictions[i:i+1], anchors.to(device), is_predictions=True
                )
                
                if len(pred_boxes_batch) > 0:
                    keep_indices = nms(
                        box_convert(pred_boxes_batch[:, 2:], "cxcywh", "xyxy"),
                        pred_boxes_batch[:, 1],
                        iou_threshold=0.3
                    )
                    pred_boxes_batch = pred_boxes_batch[keep_indices]
                
                predicted_boxes.append({
                    'boxes': pred_boxes_batch[:, 2:],
                    'scores': pred_boxes_batch[:, 1],
                    'labels': pred_boxes_batch[:, 0].long(),
                })
                
                true_boxes = cells_to_bboxes(targets[i:i+1], anchors.to(device))
                true_boxes = true_boxes[true_boxes[:, 1] == 1]
                
                target_boxes.append({
                    'boxes': true_boxes[:, 2:],
                    'labels': true_boxes[:, 0].long(),
                })
            
            metric.update(predicted_boxes, target_boxes)
    
    return metric.compute()

In [95]:
def train_yolo_detection(model, train_loader, val_loader, anchors, loss_fn=None, num_epochs=50, device='cuda'):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.001)
    if loss_fn is None:
        loss_fn = YOLOLoss()
    
    train_losses = []
    val_metrics_history = []
    best_map = 0.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_loss = 0
        num_batches = 0
        
        for batch_idx, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            predictions = model(images)
            loss = loss_fn(predictions, targets, anchors.to(device))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_train_loss = epoch_loss / num_batches
        train_losses.append(avg_train_loss)
        
        # Validation
        val_metrics = calculate_val_metrics(model, val_loader, anchors, device)
        val_metrics_history.append(val_metrics)
        
        current_map = val_metrics["map"].item()
        if current_map > best_map:
            best_map = current_map
            best_model_state = model.state_dict().copy()
        
        
        print(f'Epoch {epoch+1:2d} | Loss: {avg_train_loss:.4f} | mAP: {current_map:.4f} | mAP_0.5: {val_metrics["map_50"].item():.4f} | mAP_0.75: {val_metrics["map_75"].item():.4f}')
    
    if best_model_state is not None:
        torch.save(best_model_state, 'best_yolo_model.pth')
    
    return {
        'train_losses': train_losses,
        'val_metrics': val_metrics_history,
        'best_map': best_map
    }

In [37]:
device = 'cuda'
model = TinyYOLO(in_channels=1, num_classes=2, num_anchors=len(rescaled_anchors))
print(sum(p.numel() for p in model.parameters()))
torch.manual_seed(42)

109431


In [38]:
results = train_yolo_detection(
    model, train_loader, val_loader, rescaled_anchors, 
    num_epochs=30, device=device
)
print(f"Лучший mAP: {results['best_map']:.4f}")

Epoch  1 | Loss: 0.7719 | mAP: 0.0107 | mAP_0.5: 0.0920 | mAP_0.75: 0.0000
Epoch  2 | Loss: 0.2532 | mAP: 0.1016 | mAP_0.5: 0.4897 | mAP_0.75: 0.0000
Epoch  3 | Loss: 0.1786 | mAP: 0.2024 | mAP_0.5: 0.6871 | mAP_0.75: 0.0322
Epoch  4 | Loss: 0.1484 | mAP: 0.3004 | mAP_0.5: 0.7925 | mAP_0.75: 0.1229
Epoch  5 | Loss: 0.1292 | mAP: 0.3145 | mAP_0.5: 0.8027 | mAP_0.75: 0.1837
Epoch  6 | Loss: 0.1148 | mAP: 0.3226 | mAP_0.5: 0.7639 | mAP_0.75: 0.2214
Epoch  7 | Loss: 0.1055 | mAP: 0.3448 | mAP_0.5: 0.7975 | mAP_0.75: 0.2011
Epoch  8 | Loss: 0.0950 | mAP: 0.3396 | mAP_0.5: 0.7952 | mAP_0.75: 0.2351
Epoch  9 | Loss: 0.0846 | mAP: 0.3476 | mAP_0.5: 0.8257 | mAP_0.75: 0.2309
Epoch 10 | Loss: 0.0755 | mAP: 0.3511 | mAP_0.5: 0.8318 | mAP_0.75: 0.2474
Epoch 11 | Loss: 0.0680 | mAP: 0.3516 | mAP_0.5: 0.8100 | mAP_0.75: 0.2494
Epoch 12 | Loss: 0.0629 | mAP: 0.3594 | mAP_0.5: 0.8253 | mAP_0.75: 0.2408
Epoch 13 | Loss: 0.0561 | mAP: 0.3571 | mAP_0.5: 0.8328 | mAP_0.75: 0.2139
Epoch 14 | Loss: 0.0502 |

### Задание 2 (2 балла, необязательное). YOLOv3 loss
Мы упоминали, что на практике использовалась не совсем та же самая ошибка, что и в YOLOv3. В этом задании исправьте в классе YoloLoss ошибку регрессии, приведя её в соответствие с тем, как она описана в статье [YOLOv3: An Incremental Improvement](https://arxiv.org/abs/1804.02767) (см. раздел 2.1. Bounding Box Prediction).

Запустите обучение с изменённой ошибкой, добейтесь $mAP > 0.3$

Примечание: исправление само по себе очень небольшое, в 2 строчки кода.

In [25]:
class YOLOv3Loss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.cross_entropy = nn.CrossEntropyLoss()

    def forward(self, pred: Tensor, target: Tensor, anchors: Tensor) -> Tensor:
        # ниже входные тензоры будут меняться на месте, так что склонируем их
        pred = pred.clone()
        target = target.clone()

        # разделяем рамки на содержащие объекты и не содержащие
        # NB: ещё могут быть -1, куда отнеслось более 1 объекта - их не учитываем
        obj = target[..., 0] == 1
        no_obj = target[..., 0] == 0

        # преобразуем предсказания bbox
        scores, pred_boxes, logits = process_yolo_preds(pred, anchors)

        # no object loss: кросс-энтропия вместо MSE
        no_object_loss = self.bce(
            (scores[no_obj]),
            (target[..., 0:1][no_obj]),
        )

        # object loss: учим предсказывать IoU
        ious = intersection_over_union(pred_boxes[obj], target[..., 1:5][obj]).detach()
        object_loss = self.mse(scores[obj].sigmoid(), ious * target[..., 0:1][obj])

        # box coordinate loss: логарифмируем размеры рамок перед расчётом MSE
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        
        # здесь убрали преобразованные координаты центра, оставив сырые логиты,
        # и преобразовали target через log(p / (1 - p))
        target[..., 1:3] = torch.log(target[..., 1:3] / (1 - target[..., 1:3] + 1e-6)) 
        
        target[..., 3:5] = torch.log(1e-6 + target[..., 3:5] / anchors)
        box_loss = self.mse(pred[..., 1:5][obj], target[..., 1:5][obj])

        # class loss: здесь всё обычно
        class_loss = self.cross_entropy(logits[obj], target[..., 5][obj].long() - 1)

        # Total loss
        return box_loss + object_loss + no_object_loss + class_loss

In [39]:
device = 'cuda'
model = TinyYOLO(in_channels=1, num_classes=2, num_anchors=len(rescaled_anchors))
print(sum(p.numel() for p in model.parameters()))
torch.manual_seed(42)

109431


In [40]:
results = train_yolo_detection(
    model, train_loader, val_loader, rescaled_anchors, 
    loss_fn=YOLOv3Loss(), num_epochs=30, device=device
)
print(f"Лучший mAP: {results['best_map']:.4f}")

Epoch  1 | Loss: 2.0612 | mAP: 0.0004 | mAP_0.5: 0.0023 | mAP_0.75: 0.0000
Epoch  2 | Loss: 0.8338 | mAP: 0.0748 | mAP_0.5: 0.2743 | mAP_0.75: 0.0040
Epoch  3 | Loss: 0.5804 | mAP: 0.2575 | mAP_0.5: 0.5973 | mAP_0.75: 0.1341
Epoch  4 | Loss: 0.4228 | mAP: 0.2931 | mAP_0.5: 0.7139 | mAP_0.75: 0.1568
Epoch  5 | Loss: 0.3316 | mAP: 0.2958 | mAP_0.5: 0.7313 | mAP_0.75: 0.1645
Epoch  6 | Loss: 0.2694 | mAP: 0.2896 | mAP_0.5: 0.7835 | mAP_0.75: 0.1354
Epoch  7 | Loss: 0.2269 | mAP: 0.2661 | mAP_0.5: 0.7687 | mAP_0.75: 0.0877
Epoch  8 | Loss: 0.1957 | mAP: 0.3144 | mAP_0.5: 0.7901 | mAP_0.75: 0.1747
Epoch  9 | Loss: 0.1771 | mAP: 0.3027 | mAP_0.5: 0.7907 | mAP_0.75: 0.1334
Epoch 10 | Loss: 0.1614 | mAP: 0.3098 | mAP_0.5: 0.7920 | mAP_0.75: 0.1793
Epoch 11 | Loss: 0.1440 | mAP: 0.3284 | mAP_0.5: 0.7932 | mAP_0.75: 0.1992
Epoch 12 | Loss: 0.1312 | mAP: 0.3219 | mAP_0.5: 0.8031 | mAP_0.75: 0.1702
Epoch 13 | Loss: 0.1198 | mAP: 0.3124 | mAP_0.5: 0.7990 | mAP_0.75: 0.1298
Epoch 14 | Loss: 0.1084 |

### Задание 3 (3 балла). Выбор anchors с помощью кластеризации

В статье [YOLO9000: Better, Faster, Stronger](https://arxiv.org/abs/1612.08242) в разделе 2. Better. Dimension clusters описан способ выбора anchor boxes через кластеризацию обучающего датасета.

Проделайте то же самое с вашим обучающим датасетом, чтобы выбрать несколько anchor boxes.

В качестве результата выведите получившиеся размеры anchors для # Clusters = 5

In [28]:
def iou_wh(wh1: torch.Tensor, wh2: torch.Tensor) -> torch.Tensor:
    if wh2.dim() == 1:
        wh2 = wh2.unsqueeze(0)
    
    intersection_area = torch.min(wh1[..., 0:1], wh2[..., 0:1]) * torch.min(wh1[..., 1:2], wh2[..., 1:2])
    
    box1_area = wh1[..., 0] * wh1[..., 1]
    box2_area = wh2[..., 0] * wh2[..., 1]
    union_area = box1_area.unsqueeze(-1) + box2_area.unsqueeze(0) - intersection_area.squeeze(-1).unsqueeze(-1)
    
    iou_score = intersection_area.squeeze(-1) / union_area.squeeze(-1)
    return iou_score

In [29]:
def collect_all_wh(dataset, image_size=256):
    all_wh = []
    for i in range(len(dataset)):
        image_path = dataset.items[i]
        boxes_xyxy = torch.load(dataset.subset_dir / "bounding_boxes" / image_path.name, weights_only=True)
        if len(boxes_xyxy) == 0:
            continue
        boxes_cxcywh = box_convert(boxes_xyxy, "xyxy", "cxcywh")
        wh = boxes_cxcywh[:, 2:]
        all_wh.append(wh)
    if all_wh:
        all_wh = torch.cat(all_wh, dim=0)
    else:
        all_wh = torch.zeros(0, 2)
    return all_wh

In [30]:
def kmeans_wh(wh: torch.Tensor, k: int = 5, num_iters: int = 100, seed: int = 42) -> torch.Tensor:
    torch.manual_seed(seed)
    N = wh.shape[0]
    if N < k:
        raise ValueError(f"Number of points ({N}) less than k ({k})")
    
    idx = torch.randperm(N)[:k]
    centers = wh[idx].clone()
    
    for it in range(num_iters):
        dists = torch.zeros(N, k, device=wh.device)
        for i in range(k):
            iou = iou_wh(wh, centers[i])
            dists[:, i] = 1 - iou
        
        assignments = dists.argmin(dim=1)
        
        new_centers = torch.zeros_like(centers)
        for i in range(k):
            mask = (assignments == i)
            if mask.sum() > 0:
                new_centers[i] = wh[mask].mean(dim=0)
            else:
                new_centers[i] = centers[i]
        
        if torch.allclose(centers, new_centers, atol=1e-2):
            break
        centers = new_centers
    
    return centers

In [31]:
train_dataset = YeastDetectionDataset(
    Path("yeast_cell_in_microstructures_dataset/train"), anchors=ANCHORS, image_size=256
)

In [32]:
all_wh_pixels = collect_all_wh(train_dataset)
print(f"Общее число bboxes: {len(all_wh_pixels)}")

if len(all_wh_pixels) > 0:
    anchors_5 = kmeans_wh(all_wh_pixels, k=5)
    print("Якоря для 5 кластеров (ширина, высота):")
    for i, anchor in enumerate(anchors_5):
        print(f"Якорь {i+1}: [{anchor[0].item():.1f}, {anchor[1].item():.1f}]")

Общее число bboxes: 1118
Якоря для 5 кластеров (ширина, высота):
Якорь 1: [70.3, 105.6]
Якорь 2: [57.1, 84.9]
Якорь 3: [43.6, 48.0]
Якорь 4: [61.8, 62.1]
Якорь 5: [91.3, 88.3]


### Задание 4 (4 балла + бонусы за лучшую точность). Обучите модель


Ваша цель: $mAP > 0.6$ на валидации.

Можете использовать весь арсенал:
- использование множества якорных рамок (сформированных вручную или в результате кластеризации)
- любые изменения функции ошибки
- любые изменения архитектуры модели и регуляризация
- аугментации (вспоминаем `torchvision.transforms` и `albumentations`)
- любая длительность обучения, оптимизатор, расписание для learning rate

Бонусы за повышенную точность:
- **5 баллов**: $mAP > 0.65$
- **1 балл** за каждые следующие $0.01$ (т. е. за $mAP > 0.72$ в этом задании вы получите $4 + 12 = 16$ баллов)

**Важно**: перез запуском обучения зафиксируйте `torch.manual_seed()`

In [97]:
device = 'cuda'
model = TinyYOLO(in_channels=1, num_classes=2, num_anchors=len(rescaled_anchors))
print(sum(p.numel() for p in model.parameters()))
torch.manual_seed(42)

311443


In [98]:
results = train_yolo_detection(
    model, train_loader, val_loader, rescaled_anchors, 
    num_epochs=350, device=device   
)
print(f"Лучший mAP: {results['best_map']:.4f}")

Epoch  1 | Loss: 0.7818 | mAP: 0.0013 | mAP_0.5: 0.0067 | mAP_0.75: 0.0000
Epoch  2 | Loss: 0.3133 | mAP: 0.0739 | mAP_0.5: 0.1859 | mAP_0.75: 0.0151
Epoch  3 | Loss: 0.2459 | mAP: 0.1445 | mAP_0.5: 0.3408 | mAP_0.75: 0.0581
Epoch  4 | Loss: 0.2246 | mAP: 0.2191 | mAP_0.5: 0.4484 | mAP_0.75: 0.1693
Epoch  5 | Loss: 0.1979 | mAP: 0.2475 | mAP_0.5: 0.6157 | mAP_0.75: 0.1424
Epoch  6 | Loss: 0.1738 | mAP: 0.2987 | mAP_0.5: 0.7144 | mAP_0.75: 0.1961
Epoch  7 | Loss: 0.1495 | mAP: 0.2959 | mAP_0.5: 0.6758 | mAP_0.75: 0.2134
Epoch  8 | Loss: 0.1282 | mAP: 0.3418 | mAP_0.5: 0.8031 | mAP_0.75: 0.2116
Epoch  9 | Loss: 0.1100 | mAP: 0.3661 | mAP_0.5: 0.8279 | mAP_0.75: 0.2471
Epoch 10 | Loss: 0.0938 | mAP: 0.3480 | mAP_0.5: 0.8419 | mAP_0.75: 0.2239
Epoch 11 | Loss: 0.0785 | mAP: 0.3443 | mAP_0.5: 0.8422 | mAP_0.75: 0.2223
Epoch 12 | Loss: 0.0656 | mAP: 0.3560 | mAP_0.5: 0.8435 | mAP_0.75: 0.2340
Epoch 13 | Loss: 0.0562 | mAP: 0.3639 | mAP_0.5: 0.8243 | mAP_0.75: 0.2617
Epoch 14 | Loss: 0.0480 |